# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## 1. PromptTemplate - 단일 문자열 템플릿

In [2]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"})
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [4]:
import sys
sys.path.append("..")
from common_config import llm_connect
model ="gpt-5.4"

temperature = 0
max_tokens = 2086

llm = llm_connect(model=model)
# llm = llm_connect(model=model, temperature=temperature, max_tokens=max_tokens)

messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)

과적합(overfitting)은 **모델이 학습 데이터에 너무 딱 맞게 배워서**, 새로운 데이터에서는 성능이 떨어지는 현상입니다.

### 쉽게 말하면
시험문제를 **이해해서 푼 게 아니라 외워버린 상태**와 비슷합니다.

- **학습 데이터**에서는 점수가 아주 높음
- 그런데 **처음 보는 데이터**에서는 잘 못 맞춤

### 왜 생기나요?
주로 이런 경우에 발생합니다:

- 모델이 너무 복잡할 때
- 학습 데이터가 너무 적을 때
- 데이터의 잡음(noise)까지 학습했을 때
- 너무 오래 학습했을 때

### 예시
예를 들어 고양이/강아지 사진을 구분하는 모델이 있는데,
고양이 사진 배경이 항상 빨간색이었다고 해봅시다.

그러면 모델이 **고양이의 특징**이 아니라
“배경이 빨간색이면 고양이”라고 잘못 학습할 수 있습니다.

이 경우 학습 데이터에서는 잘 맞지만,
배경이 다른 새로운 사진에서는 틀리게 됩니다.

### 과적합의 징후
보통 이런 모습으로 나타납니다:

- 훈련 성능은 계속 좋아짐
- 검증/테스트 성능은 어느 순간부터 나빠짐

### 해결 방법
대표적으로는:

- 데이터를 더 많이 모으기
- 데이터 증강(data augmentation)
- 모델을 단순하게 만들기
- 정규화(regularization) 사용
- 드롭아웃(dropout) 사용
- 조기 종료(early stopping)
- 교차 검증(cross validation)

원하시면 제가 **과적합 / 과소적합 / 일반화** 차이도 같이 설명해드릴게요.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [11]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

In [12]:
messages

ChatPromptValue(messages=[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={})])

In [ ]:
response = llm.invoke(messages)
print(response.content)

In [10]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님이세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [13]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.


## 문제 해결

In [15]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "좋은 오후 입니다.."
})

response = llm.invoke(messages)
print(response.content)

한국어: 좋은 오후 입니다..
영어: Good afternoon.
